<h2>ЛР 2 Проведение экспериментов по настройке модели

In [10]:
import os

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, TargetEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from catboost import CatBoostClassifier

from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

<h3>Разбиваем выборку

In [2]:
df = pd.read_pickle('../data/clean_data.pkl')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Major_Category              50000 non-null  category
 1   Year_of_Study               50000 non-null  category
 2   Pre_Semester_GPA            50000 non-null  float32 
 3   Weekly_GenAI_Hours          50000 non-null  float32 
 4   Primary_Use_Case            50000 non-null  category
 5   Prompt_Engineering_Skill    50000 non-null  category
 6   Tool_Diversity              50000 non-null  int8    
 7   Paid_Subscription           50000 non-null  bool    
 8   Traditional_Study_Hours     50000 non-null  float32 
 9   Perceived_AI_Dependency     50000 non-null  int8    
 10  Institutional_Policy        50000 non-null  category
 11  Anxiety_Level_During_Exams  50000 non-null  int8    
 12  Post_Semester_GPA           50000 non-null  float32 
 13  Skill_Retention_

In [3]:
# df = df.rename(columns={'Burnout_Risk_Level' : 'target'}) - риск выгорания ["Low", "Medium", "High"] таргет
df = df.drop(columns=['Major_Category', 'AI_Usage_Group', 'High_Risk_Student', 'Academic_Perfomance_Drop'])
df

,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,target,GPA_Delta,Total_Stud_Hours,AI_Usage_Percentage
0,Senior,2.418,23.309999,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.440002,High,-0.025,31.439999,0.741412
1,Junior,3.821,1.120000,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.389999,Low,-0.125,17.770000,0.063028
2,Freshman,3.398,21.260000,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.930000,Medium,0.101,31.610001,0.672572
3,Senior,3.789,1.820000,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.580002,Medium,0.211,17.049999,0.106745
4,Sophomore,3.635,9.290000,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.000000,Medium,0.163,21.840000,0.425366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,Senior,2.899,12.160000,Copywriting/Drafting,Beginner,2,False,13.36,2,Allowed_With_Citation,2,3.584,66.160004,High,0.685,25.520000,0.476489
49996,Senior,2.870,2.510000,Copywriting/Drafting,Intermediate,1,False,4.67,3,Actively_Encouraged,3,3.096,81.620003,Medium,0.226,7.180000,0.349582
49997,Senior,3.177,15.870000,Summarizing_Reading,Advanced,5,True,3.92,4,Allowed_With_Citation,5,3.605,97.209999,High,0.428,19.790001,0.801920
49998,Junior,3.398,19.910000,Debugging/Troubleshooting,Intermediate,5,False,7.10,5,Allowed_With_Citation,3,3.527,84.120003,Medium,0.129,27.010000,0.737134


In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1), df['target'], test_size=0.25, random_state=2)

In [5]:
X_train.iloc[0]

Year_of_Study                                Senior
Pre_Semester_GPA                              3.109
Weekly_GenAI_Hours                             5.85
Primary_Use_Case               Copywriting/Drafting
Prompt_Engineering_Skill                   Beginner
Tool_Diversity                                    3
Paid_Subscription                              True
Traditional_Study_Hours                       11.16
Perceived_AI_Dependency                           4
Institutional_Policy          Allowed_With_Citation
Anxiety_Level_During_Exams                        4
Post_Semester_GPA                              3.32
Skill_Retention_Score                     57.900002
GPA_Delta                                     0.211
Total_Stud_Hours                              17.01
AI_Usage_Percentage                        0.343915
Name: 45019, dtype: object

In [6]:
cat_features = X_train.select_dtypes(include=['category','object']).columns.to_list()
cat_features

['Year_of_Study',
 'Primary_Use_Case',
 'Prompt_Engineering_Skill',
 'Institutional_Policy']

In [7]:
num_features = X_train.select_dtypes(include=['number']).columns.to_list()
num_features

['Pre_Semester_GPA',
 'Weekly_GenAI_Hours',
 'Tool_Diversity',
 'Traditional_Study_Hours',
 'Perceived_AI_Dependency',
 'Anxiety_Level_During_Exams',
 'Post_Semester_GPA',
 'Skill_Retention_Score',
 'GPA_Delta',
 'Total_Stud_Hours',
 'AI_Usage_Percentage']

In [12]:
s_scaler = StandardScaler()
t_encoder = TargetEncoder()
classificator = CatBoostClassifier()
t_encoder

,"categories categories: ""auto"" or list of shape (n_features,) of array-like, default=""auto""Categories (unique values) per feature:- `""auto""` : Determine categories automatically from the training data.- list : `categories[i]` holds the categories expected in the i-th column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories are stored in the `categories_` fitted attribute.",'auto'
,"target_type target_type: {""auto"", ""continuous"", ""binary"", ""multiclass""}, default=""auto""Type of target.- `""auto""` : Type of target is inferred with :func:`~sklearn.utils.multiclass.type_of_target`.- `""continuous""` : Continuous target- `""binary""` : Binary target- `""multiclass""` : Multiclass target.. note:: The type of target inferred with `""auto""` may not be the desired target type used for modeling. For example, if the target consisted of integers between 0 and 100, then :func:`~sklearn.utils.multiclass.type_of_target` will infer the target as `""multiclass""`. In this case, setting `target_type=""continuous""` will specify the target as a regression problem. The `target_type_` attribute gives the target type used by the encoder... versionchanged:: 1.4 Added the option 'multiclass'.",'auto'
,"smooth smooth: ""auto"" or float, default=""auto""The amount of mixing of the target mean conditioned on the value of thecategory with the global target mean. A larger `smooth` value will putmore weight on the global target mean.If `""auto""`, then `smooth` is set to an empirical Bayes estimate.",'auto'
,"cv cv: int, cross-validation generator or an iterable, default=5Determines the splitting strategy used in the internal :term:`cross fitting`during :meth:`fit_transform`. Splitters where each sample index doesn't appearin the validation fold exactly once, raise a `ValueError`.Possible inputs for cv are:- `None`, to use a 5-fold cross-validation chosen internally based on `target_type`,- integer, to specify the number of folds for the cross-validation chosen internally based on `target_type`,- :term:`CV splitter` that does not repeat samples across validation folds,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if `target_type` is `""continuous""`, :class:`KFold` isused, otherwise :class:`StratifiedKFold` is used.Refer :ref:`User Guide <cross_validation>` for more information oncross-validation strategies... versionchanged:: 1.9 Cross-validation generators and iterables can also be passed as `cv`.",5
,"shuffle shuffle: bool, default=TrueWhether to shuffle the data in :meth:`fit_transform` before splitting intofolds. Note that the samples within each split will not be shuffled. Onlyapplies if `cv` is an int or `None`. If `cv` is a cross-validation generator oran iterable, `shuffle` is ignored... deprecated:: 1.9 `shuffle` is deprecated and will be removed in 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling instead.",'deprecated'
,"random_state random_state: int, RandomState instance or None, default=NoneWhen `shuffle` is True, `random_state` affects the ordering of theindices, which controls the randomness of each fold. Otherwise, thisparameter has no effect.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`... deprecated:: 1.9 `random_state` is deprecated and will be removed in 1.11. Pass a cross-validation generator as `cv` argument to specify the random state of the shuffling instead.",'deprecated'
